In [ ]:
# Imports

import cv2
import pandas as pd
import numpy as np
import scipy
from matplotlib import pyplot as plt
import matplotlib
from scipy import ndimage
import matplotlib.patches as patches
import math
from pathlib import Path
import yaml

In [ ]:
#Define the path to the folder into which you unpacked the Zip archive with the large image files
external_input_path = Path('../External_Image_Inputs')

# Define the sample ID; for example '02', '7b', 10' for Patient 2, Patient 7b and Patient 10, respectively
sample_id = '01'

# Define the amount of pixels per µm in the visium image, the spot radius and the mapping radius
# For the mechanoMaST paper, 1.3633 px/µm (corresponding to the visium image), a spot_radius_um of 27.5 and a mapping_radis_um of 50 was used.
px_per_um = 1.3633
spot_radius_um = 27.5
mapping_radius_um = 50

spot_radius = round(spot_radius_um * px_per_um)
mapping_radius = math.ceil(mapping_radius_um * px_per_um)

#define the path to the config file and load the data for the selected sample
config_path = Path('../config_samples.yaml')
with config_path.open("r") as f:
    config = yaml.safe_load(f)

sample = config[sample_id]    

In [ ]:
# Image input paths
vis_img_path = external_input_path / 'Images' / f'Visium_Section_HE_Tilescans/Patient{sample_id}_visium-section_10Xtile_HE.ome.tif'
afm_img_path = external_input_path / 'Images' / f'AFM_Section_HE-Tilescans/Patient{sample_id}_afm-section_10Xtile_HE.ome.tif'
he_20x_paths = sorted((external_input_path / 'Images' / f'AFM_Section_HE_20X').glob(f'Patient{sample_id}*.tif'))
hoechst_20x_paths = sorted((external_input_path / 'Images' / f'AFM_Section_Hoechst_20X').glob(f'Patient{sample_id}*.tif'))

# Matrix input paths
he_matrix_path = Path(f'../Inputs/Matrices/AFM-section_to_Visium-section/Patient{sample_id}_matrix.txt')
hoechst_matrix_paths = sorted((Path('../Inputs/Matrices/Hoechst-FOV_to_HE-FOV').glob(f'Patient{sample_id}*.txt')))

# Tissue position file input path
tissue_pos_path = Path(f'../Inputs/tissue_positions.csv')

# AFM measurements path
afm_path = Path('../Inputs/AFM_Measurements.xlsx')

# Output paths
output_path = Path('../Outputs/')

In [ ]:
#Optional cell for quality control
#Check that there is an equal number of paths

if len(he_20x_paths) == len(hoechst_20x_paths) == len(hoechst_matrix_paths):
    print(str(len(he_20x_paths)) + " FOVs found.") 
else:
    print("Differing amounts of FOVs detected. Please double-check the inputs.")

In [ ]:
# Load images, matrices, tissue positions and afm measurements
visium_img = cv2.imread(vis_img_path)
afm_tile_img = cv2.imread(afm_img_path)
he_20x_imgs = [cv2.imread(path) for path in he_20x_paths]
hoechst_20x_imgs = [cv2.imread(path) for path in hoechst_20x_paths]

he_matrix = np.matrix(np.loadtxt(he_matrix_path, delimiter=","))
hoechst_matrices = [np.matrix(np.loadtxt(matrix_path, delimiter=",")) for matrix_path in hoechst_matrix_paths]

all_tissue_pos_df =  pd.read_csv(tissue_pos_path)
tissue_pos_df = all_tissue_pos_df[all_tissue_pos_df["sample"] == f"Patient{sample_id}"]

all_afm_values_df =  pd.read_excel(afm_path)
afm_df = all_afm_values_df[all_afm_values_df["Sample"] == f"Patient{sample_id}"]

# For one of the samples (Patient 1) there was a rotation of 18 degrees between acquiring the H&E images of the FOVs and the H&E tilescan
# which needs to be accomodated
if sample.get("image_rotation") is not None:
    afm_tile_img = ndimage.rotate(afm_tile_img, sample['image_rotation'], reshape = False)
    w,h,_ = afm_tile_img.shape
    rot_matrix = cv2.getRotationMatrix2D((w/2, h / 2), sample['image_rotation'], 1)
    rot_matrix = np.vstack([rot_matrix, [0, 0, 1]])


In [ ]:
# Functions used in this Notebook

# apply template matching to image
def template_matching(img, tmpl, sf=0.5, display=False):
    
    # determine new dimensions for the template and resize it
    # .shape outputs [h, w, c] cv2.resize needs (w, h)
    h, w, _ = tmpl.shape
    new_dim = (int(w * sf), int(h * sf))
    tmpl = cv2.resize(tmpl, new_dim, interpolation=cv2.INTER_AREA)

    # perform the template matching and get parameters for best match
    # max_loc holds the top left coordinates of the best match
    res = cv2.matchTemplate(img, tmpl, cv2.TM_CCOEFF_NORMED)
    _, _, _, max_loc = cv2.minMaxLoc(res)

    trans_matrix = np.matrix([[sf, 0, max_loc[1]],
                              [0, sf, max_loc[0]],
                              [0, 0, 1]])

    if display:
        top_left = max_loc
        bottom_right = (top_left[0] + int(w * sf), top_left[1] + int(h * sf))
        rect_img = img.copy()
        cv2.rectangle(rect_img, top_left, bottom_right, (255, 0, 255), 100)
        fov = img[top_left[1]:bottom_right[1], top_left[0]:bottom_right[0]]

        plt.subplot(131), plt.imshow(cv2.cvtColor(tmpl, cv2.COLOR_BGR2RGB))
        plt.title('Template'), plt.xticks([]), plt.yticks([])
        plt.subplot(132), plt.imshow(cv2.cvtColor(fov, cv2.COLOR_BGR2RGB))
        plt.title('Found FOV'), plt.xticks([]), plt.yticks([])
        plt.subplot(133), plt.imshow(cv2.cvtColor(rect_img, cv2.COLOR_BGR2RGB))
        plt.title('Detected Point'), plt.xticks([]), plt.yticks([])

        plt.show()

    return trans_matrix


# affines images via a transformation matrix
def affine_image(img, output_shape, trans_matrix):
    
    # split channels
    channel_list = cv2.split(img)

    # affine each channel individually
    affined_list = [scipy.ndimage.affine_transform(channel, trans_matrix, output_shape=output_shape)
                    for channel in channel_list]

    # merge channels
    affined_img = cv2.merge(affined_list)

    return affined_img




# adds an alpha channel to a three-channel image that sets all the black pixels to transparent
def remove_background(img):
    
    # create an array with True for every non-zero pixel and False for every black pixel 
    # multiplying with 255 creates an alpha channel setting each black pixel transparent, everything else opaque
    alpha_channel = (img != [0, 0, 0]).any(axis=-1) * 255
    
    # add the alpha channel to the input image
    img_without_background = np.concatenate([img, alpha_channel[..., np.newaxis]], axis=-1)
    
    return img_without_background.astype(np.uint8)




# Reproduce the coordinate grid from the AFM measurements. Here, the AFM recorded in a snake-like pattern.
def create_coordinate_grid(grid_dim=(10, 10), grid_size=(200, 200), center_coords=(1024, 1024), pw=0.3337, x_step=None):
    x_step = grid_size[0] / grid_dim[0] / pw
    y_step = grid_size[1] / grid_dim[1] / pw

    min_x = center_coords[0] - ((x_step * (grid_dim[0] - 1))) / 2
    max_x = center_coords[0] + ((x_step * (grid_dim[0] - 1))) / 2
    min_y = center_coords[1] - ((y_step * (grid_dim[1] - 1))) / 2
    max_y = center_coords[1] + ((y_step * (grid_dim[1] - 1))) / 2

    x_coord = max_x + x_step

    grid_coords = []

    for counter, i in enumerate(range(grid_dim[0])):
        x_coord -= x_step
        
        if counter % 2 == 0:  # alternating direction between even and odd columns
            y_coord = max_y
            
            for j in range(grid_dim[1]):
                grid_coords.append(np.array([x_coord, y_coord]))
                y_coord -= y_step
            
        else:
            y_coord = min_y
            
            for j in range(grid_dim[1]):
                grid_coords.append(np.array([x_coord, y_coord]))
                y_coord += y_step
            

    return np.array(grid_coords)
    #



# transform AFM coordinates via a napari-generated transformation matrix
def transform_coords(coords, matrix):

    coord_matrix = matrix.copy()
    coord_matrix[0,2], coord_matrix[1,2] = coord_matrix[1,2], coord_matrix[0,2] 
    coord_matrix[0,0], coord_matrix[1,1] = coord_matrix[1,1], coord_matrix[0,0]
    coord_matrix[0,1], coord_matrix[1,0] = coord_matrix[1,0], coord_matrix[0,1]
    
    # Add a one to each AFM coordinate to make them compatible with the transformation matrix
    coords = np.column_stack([coords, np.ones(coords.shape[0])])
    
    # multiply coords with transformation matrix and return transformed coords
    return np.array(np.vstack([np.dot(coord_matrix, coord)[:,:2] for coord in coords]))


In [50]:
# optional: Affine the AFM tile image to the visium image (for visualization and QC purposes only)

if sample.get("image_rotation") is None:
    he_affined = affine_image(afm_tile_img, visium_img.shape[:2], np.linalg.inv(he_matrix))
    

else: #for rotated samples (Patient 1)

    he_affined = affine_image(ndimage.rotate(afm_tile_img, 360 - sample['image_rotation'], reshape = False), visium_img.shape[:2], np.linalg.inv(he_matrix))
plt.imshow(cv2.cvtColor(he_affined, cv2.COLOR_BGR2RGB))


# optional: save affined image
#cv2.imwrite(enter/saving/path/here.tif, he_affined)

In [ ]:
crop = sample.get('template_cropping') #for Patient06 FOV1 since it is close to the image border

if crop: 
    crop_fov = crop.get('crop_FOV')
    crop_index = crop_fov - 1

    top = crop.get("top", 0)
    bottom = crop.get("bottom")
    left = crop.get("left", 0)
    right = crop.get("right")

    crop_tmpl = he_20x_imgs[crop_index][top:bottom, left:right]


    plt.subplot(121), plt.imshow(cv2.cvtColor(he_20x_imgs[crop_index], cv2.COLOR_BGR2RGB))
    plt.title('Original_FOV' + str(crop_fov)), plt.xticks([]), plt.yticks([])
    plt.subplot(122), plt.imshow(cv2.cvtColor(crop_tmpl, cv2.COLOR_BGR2RGB))
    plt.title('Cropped_FOV' + str(crop_fov)), plt.xticks([]), plt.yticks([])
    plt.show()

    adapted_template_list = he_20x_imgs.copy()
    adapted_template_list[crop_index] = crop_tmpl

    tm_matrices = [template_matching(afm_tile_img, he_20x_img, sf = 0.5, display = True)
                  for he_20x_img in adapted_template_list]

    crop_matrix = np.matrix([[1, 0, -top],
                              [0, 1, -left],
                              [0, 0, 1]])

    tm_matrices[crop_index] = tm_matrices[crop_index] @ crop_matrix

else:
    tm_matrices = [template_matching(afm_tile_img, he_20x_img, sf = 0.5, display = True)
              for he_20x_img in he_20x_imgs]

In [ ]:
# Save the template matching transformation matrices
tm_matrix_save_path = output_path / 'Matrices' / 'Template_Matching' 
tm_matrix_save_path.mkdir(parents=True, exist_ok=True)

[np.savetxt(tm_matrix_save_path  / f'Patient{sample_id}_FOV{i + 1:02d}.txt', np.asarray(matrix), delimiter=',')
 for i, matrix in enumerate(tm_matrices)]

In [ ]:
# compute direct transformation matrices

if sample.get("image_rotation") is None:
    matrices = [he_matrix @ tm_matrix @ hoechst_matrix 
                for tm_matrix, hoechst_matrix in zip(tm_matrices, hoechst_matrices)]
    
else: #exception for Patient 1 which has a 18 degree rotation
    he_matrix_rot = np.dot(he_matrix, rot_matrix)
    matrices = [he_matrix_rot @ tm_matrix @ hoechst_matrix 
                    for tm_matrix, hoechst_matrix in zip(tm_matrices, hoechst_matrices)]

In [ ]:
# Save the direct transformation matrices
direct_matrix_save_path = output_path / 'Matrices' / 'Direct_Transformation' 
direct_matrix_save_path.mkdir(parents=True, exist_ok=True)

[np.savetxt(direct_matrix_save_path  / f'Patient{sample_id}_FOV{i + 1:02d}.txt', np.asarray(matrix), delimiter=',')
 for i, matrix in enumerate(matrices)]

In [ ]:
# This cell is optional and for quality control purposes.

# Affine the Hoechst FOV images to the Visium image to ensure a proper mapping
affined_hoechst_imgs = [affine_image(img, visium_img.shape[:2], np.linalg.inv(matrix)) 
                for img, matrix in zip(hoechst_20x_imgs, matrices)]

# remove the black background for visualization purposes
affined_imgs_no_bg = [remove_background(img) for img in affined_hoechst_imgs]

# create an overlay image 
overlay_img = visium_img.copy()

for img in affined_imgs_no_bg:

    # Extract alpha channel from overlay image
    alpha_channel = img[:, :, 3]

    # Create a mask for non-zero alpha values
    mask = alpha_channel > 0

    # Copy overlay onto the background using the mask
    overlay_img[mask] = img[mask, :3]

In [ ]:
# If you ran the previous cell, you can display the created overlay image here.
plt.figure(figsize = (10,10))
plt.imshow(cv2.cvtColor(overlay_img, cv2.COLOR_BGRA2RGBA))
plt.show()

#optional: save the overlay image
#cv2.imwrite(enter/saving/path/here.tif, overlay_img)

#optional: save the affined images without background
#affined_saving_path = 'enter/saving/path/here' 
#[cv2.imwrite(affined_saving_path / f'Patient{sample_id}_FOV{i + 1:02d}_affined.tif', img) for i, img in enumerate(affined_imgs_no_bg)]

In [ ]:
# load the grid dimensions from the config file
grid_dimensions = sample["grid_dimensions"]
# calculate the corresponding grid sizes in µm, here measurements were spaced 20 µm apart
grid_sizes = [[x * 20 for x in y] for y in grid_dimensions]

# create list of standard grids, the create coordinate grid function assumes snake-like measurement placement
standard_grids = [create_coordinate_grid(grid_dim=dim, grid_size=size) for dim, size in zip(grid_dimensions, grid_sizes)]

In [ ]:
# Optional cell for quality control
# Display the first created afm_grid on top of the first FOV to ensure the measurement order is correct.
# In case you need to make changes to the grid creation function to adapt it to your AFM, 
# you can check here, that it is correct. In our case, the AFM started all measurements at the bottom right.
# The comments regarding the colours refer to a 10x10 grid

plt.figure(figsize = (10,10))
plt.imshow(cv2.cvtColor(hoechst_20x_imgs[0], cv2.COLOR_BGR2RGB))
plt.scatter(x = standard_grids[0][:,0], y= standard_grids[0][:,1], c = 'w') #everything white
plt.scatter(x = standard_grids[0][0,0], y= standard_grids[0][0,1], c = 'r') #first coordinate red
plt.scatter(x = standard_grids[0][1:10,0], y= standard_grids[0][1:10,1], c = 'b') #rest of first collumn blue
plt.scatter(x = standard_grids[0][10,0], y= standard_grids[0][10,1], c = 'y') #first of second column yellow
plt.scatter(x = standard_grids[0][20,0], y= standard_grids[0][20,1], c = 'm') #first of third column magenta
plt.scatter(x = standard_grids[0][-1,0], y= standard_grids[0][-1,1], c = 'g') #last coordinate green

plt.show()

In [ ]:
# Perform coordinate transformations
transformed_coords = [transform_coords(standard_grid, matrix) 
                      for standard_grid, matrix in zip(standard_grids, matrices)]

In [ ]:
# Optional cell for quality control. Only works if the optional cell to create the overlay image was executed.
# Display the transformed coordinates on top of the overlay image to make sure they are mapped to the FOVs

plt.figure(figsize = (30,30))
plt.imshow(cv2.cvtColor(overlay_img, cv2.COLOR_BGRA2RGBA))
for coordlist in transformed_coords:
    plt.scatter(x = coordlist[:,0], y= coordlist[:,1], c = 'w') #everything white
    
plt.show()

In [ ]:
# Filter out only spots over tissue, extract coordinates
tissue_pos_df = tissue_pos_df[tissue_pos_df["in_tissue"] == 1]
spot_coords = tissue_pos_df[['pxl_col_in_fullres', 'pxl_row_in_fullres']].values

# If the image is rotated 180 degrees the coordinates need to be transformed
if sample["spot_position_rotation"]: 
    height, width, _ = visium_img.shape
    spot_coords = np.column_stack((width - spot_coords[:, 0], height - spot_coords[:, 1]))

# add transformed coordinates to the dataframe
tissue_pos_df['x_cord_trans'] = spot_coords[:,0]
tissue_pos_df['y_cord_trans'] = spot_coords[:,1]

In [187]:
# This cell is optional: Test that the tissue positions are within the sample
spot_img = visium_img.copy()

# draw circles at the corresponding coordinates
for (x, y) in spot_coords: # 37 px radius: 55µm diameter visium spot * 1.3633 px/µm /2
    cv2.circle(spot_img, (x, y),37, (255,255,255), 10)  # -1 means fill the circle

# display image
plt.figure(figsize = (30,30))
plt.imshow(cv2.cvtColor(spot_img, cv2.COLOR_BGR2RGB))

In [ ]:
#Filter the tissue positons dataframe to only include spots that are close to the AFM grid

filtered_dfs = []

for grid in transformed_coords:
    
    # Find the extreme coords in x and y directions
    min_x, min_y = np.min(grid, axis=0)
    max_x, max_y = np.max(grid, axis=0)
    
    working_df = tissue_pos_df[(tissue_pos_df['x_cord_trans'] > min_x - mapping_radius) &
                            (tissue_pos_df['x_cord_trans'] < max_x + mapping_radius) &
                           (tissue_pos_df['y_cord_trans'] > min_y - mapping_radius) &
                           (tissue_pos_df['y_cord_trans'] < max_y + mapping_radius)].copy()
    filtered_dfs.append(working_df)

filtered_df = pd.concat(filtered_dfs)

#optional: display head of the dataframe
filtered_df.head()

In [ ]:
# map ST spot to each AFM coord
AFM_index = []
spot_x = []
spot_y = []
distances = []
barcode = []
grids = []
afm_x = []
afm_y = []


#compute distance between AFM coordinates and ST spot centres until an AFM measurement is located within the mapping radius of a spot
for grid, df, coordlist in zip(range(1,len(filtered_dfs)+1), filtered_dfs, transformed_coords):
    filtered_barcodes = df['barcode'].values
    for i, coord in enumerate(coordlist):
        for j, spot in enumerate(zip(df['x_cord_trans'],df['y_cord_trans'])):
            distance = np.sqrt((coord[0] - spot[0])**2 + (coord[1] - spot[1])**2)/px_per_um 
            if distance <= mapping_radius_um:
                print(i, 'grid' + '%02d' % grid, coord, distance/px_per_um, spot, filtered_barcodes[j]) #optional print statement for QC
                AFM_index.append(i)
                afm_x.append(coord[0])
                afm_y.append(coord[1])
                spot_x.append(spot[0])
                spot_y.append(spot[1])
                distances.append(distance) #The distance will be recorded in µm
                barcode.append(filtered_barcodes[j])
                grids.append('grid' + '%02d' % grid)
                break

In [ ]:
#create mapping dataframe
mapping_df = pd.DataFrame({'Grid': grids, 'Position_Index': AFM_index, 'Spot X': spot_x, 'Spot Y': spot_y, 
                           'AFM X': afm_x, 'AFM Y': afm_y, 'Distance': distances, 'Barcode': barcode})
mapping_df.head()

In [ ]:
mapped_df = pd.merge(afm_df, mapping_df, on=['Position_Index', 'Grid'])

In [ ]:
# Save the mapped dataframe
mapped_df_save_path = output_path / 'Mapping'
mapped_df_save_path.mkdir(parents=True, exist_ok=True)

mapped_df.to_csv(
    mapped_df_save_path / f'Patient{sample_id}_ST_AFM_Mapping.csv', index=False)

In [ ]:
#Optional cell: Show the stiffness values on top of the Visium image

vmin = mapped_df["Young's_modulus_[Pa]"].min()
vmax = mapped_df["Young's_modulus_[Pa]"].max()


for grid in mapped_df['Grid'].unique():
        plot_data = mapped_df[mapped_df['Grid'] == grid]
        pad = 50
        
        # Determine crop boundaries
        xmin = int(plot_data["Spot X"].min() - pad)
        xmax = int(plot_data["Spot X"].max() + pad)
        ymin = int(plot_data["Spot Y"].min() - pad)
        ymax = int(plot_data["Spot Y"].max() + pad)

        # Crop image
        image_crop = visium_img[ymin:ymax, xmin:xmax]

        # Create figure AND axes
        fig, ax = plt.subplots(figsize=(10, 10))

        # Show cropped image in original coordinate system
        ax.imshow(cv2.cvtColor(image_crop,cv2.COLOR_BGR2RGB),
                extent=[xmin, xmax, ymax, ymin]
        )

        # Plot the points
        sc = ax.scatter(
                plot_data['AFM X'],
                plot_data['AFM Y'],
                c=plot_data["Young's_modulus_[Pa]"],
                edgecolors='k',
                s= 100,
                cmap='viridis',
                norm=matplotlib.colors.LogNorm(vmin, vmax)
        )

        # Set limits
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymax, ymin)

        # Remove x/y ticks
        ax.set_xticks([])
        ax.set_yticks([])

        # Colorbar with label
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label("Young's modulus [Pa]")

        ax.set_title(f"Patient{sample_id}_{grid}")

        plt.show()